# Process the SPARC data

Convert the three MRT sources to CSV in `data/raw/`, then build the selected
galaxy properties and rotation-curve tables from those CSVs.

The sample excludes quality flag 3 and requires the full inclination uncertainty
interval to fall within 30–80 degrees. Stellar masses use disk and bulge mass-to-light
ratios of 0.5 and 0.7 at 3.6 microns.

In [ ]:
from pathlib import Path
import sys

# Locate the repository from either its root or notebooks/.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "utils" / "data_loading.py").is_file()
     and (path / "data" / "raw").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch this notebook from within the project directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import display
from utils.data_loading import (
    load_bulge_catalog,
    load_galaxy_catalog,
    load_galaxy_properties,
    load_rotation_curves,
)

## Convert the MRT sources to CSV

In [ ]:
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
GALAXY_PROPERTIES_PATH = RAW_DATA_DIR / "T1SPARC_Lelli2016c.csv"
ROTATION_CURVES_PATH = RAW_DATA_DIR / "T2MassModels_Lelli2016c.csv"
BULGE_LUMINOSITIES_PATH = RAW_DATA_DIR / "Bulges.csv"

for csv_path, loader in (
    (GALAXY_PROPERTIES_PATH, load_galaxy_catalog),
    (ROTATION_CURVES_PATH, load_rotation_curves),
    (BULGE_LUMINOSITIES_PATH, load_bulge_catalog),
):
    table = loader(csv_path.with_suffix(".mrt"))
    table.to_csv(csv_path, index=False)
    print(f"Saved {len(table):,} rows to {csv_path}")

## Select galaxies and calculate stellar masses

In [ ]:
INCLINATION_MIN = 30
INCLINATION_MAX = 80

galaxy_properties = load_galaxy_properties(
    GALAXY_PROPERTIES_PATH,
    bulge_luminosities_path=BULGE_LUMINOSITIES_PATH,
    inclination_min=INCLINATION_MIN,
    inclination_max=INCLINATION_MAX,
)
rotation_curve_measurements = load_rotation_curves(ROTATION_CURVES_PATH)
display(galaxy_properties.head())

## Join the rotation measurements to the selected sample

In [ ]:
selected_galaxy_rotation_curves = rotation_curve_measurements.merge(
    galaxy_properties.drop(columns="stellar_mass_msun"),
    on="galaxy",
    how="inner",
    validate="many_to_one",
)
print(
    f"{len(galaxy_properties):,} selected galaxies; "
    f"{len(selected_galaxy_rotation_curves):,} selected measurements"
)
display(selected_galaxy_rotation_curves.head())
disk_sizes = load_galaxy_catalog(GALAXY_PROPERTIES_PATH)[["galaxy", "disk_scale_length_kpc"]]
coverage = selected_galaxy_rotation_curves[["galaxy", "radius"]].merge(
    disk_sizes, on="galaxy", validate="many_to_one"
)
coverage["inner_point"] = coverage.radius <= 2 * coverage.disk_scale_length_kpc
coverage = coverage.groupby("galaxy", as_index=False).agg(
    disk_scale_length_kpc=("disk_scale_length_kpc", "first"),
    innermost_radius_kpc=("radius", "min"),
    inner_point_count=("inner_point", "sum"),
)
if not coverage.disk_scale_length_kpc.gt(0).all():
    raise ValueError("Positive disk scale lengths are required for inner coverage")
coverage["innermost_radius_over_disk_scale"] = (
    coverage.innermost_radius_kpc / coverage.disk_scale_length_kpc
)
galaxy_properties = galaxy_properties.merge(coverage, on="galaxy", how="left", validate="one_to_one")

## Save the processed tables

The properties table stores stellar mass and inner rotation-curve coverage per galaxy.
The rotation-curve table keeps one row per radius measurement for fitting.

In [ ]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
for filename, table in {
    "selected_galaxy_properties.csv": galaxy_properties,
    "selected_galaxy_rotation_curves.csv": selected_galaxy_rotation_curves,
}.items():
    output_path = PROCESSED_DATA_DIR / filename
    table.to_csv(output_path, index=False, lineterminator="\r\n")
    print(f"Saved {len(table):,} rows to {output_path}")